In [ ]:
!pip install -q crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/4

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

In [ ]:
from crewai import LLM

llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.2
)

In [ ]:
from crewai.flow.flow import Flow, start, listen, router, or_

class SupportFlow(Flow):

    @start()
    def receive_ticket(self):
        print(self.state["ticket"])

    @listen(receive_ticket)
    def categorize_ticket(self):
        answer = llm.call(
            "Classify this support ticket into one category:\n"
            "- refund: the customer wants their money back for an order or purchase\n"
            "- billing: charges, invoices, payments or subscription costs\n"
            "- account_access: login, password or account problems\n"
            "- other: anything else\n"
            "Reply with one word only.\n\nTicket: " + self.state["ticket"]
        )
        self.state["category"] = answer.strip().lower()
        print(self.state["category"])

    @router(categorize_ticket)
    def route_by_category(self):
        if self.state["category"] in ["billing", "account_access", "refund"]:
            return self.state["category"]
        return "other"

    @listen("billing")
    def handle_billing(self):
        self.state["team"] = "Billing team"
        answer = llm.call(
            "Is this billing ticket urgent (money lost, double charge, failed "
            "payment) or not urgent (general question)? "
            "Reply with one word: high or low.\n\n"
            "Ticket: " + self.state["ticket"]
        )
        self.state["priority"] = answer.strip().lower()

    @listen("account_access")
    def handle_access(self):
        self.state["team"] = "Security team"
        answer = llm.call(
            "Is this login/account ticket urgent (hacked account, locked out, "
            "cannot access at all) or not urgent (general question)? "
            "Reply with one word: high or low.\n\n"
            "Ticket: " + self.state["ticket"]
        )
        self.state["priority"] = answer.strip().lower()

    @listen("refund")
    def handle_refund(self):
      self.state["team"] = "Billing team"
      self.state["priority"] = "high"

    @listen("other")
    def handle_other(self):
        self.state["team"] = "General support"
        self.state["priority"] = "normal"

    @listen(or_(handle_billing, handle_access, handle_refund, handle_other))
    def resolve_ticket(self):
        reply = llm.call(
            "Write a short, polite 2-line reply to this customer. Tell them their "
            "ticket was sent to the " + self.state["team"] + " with "
            + self.state["priority"] + " priority.\n\nTicket: " + self.state["ticket"]
        )
        self.state["resolution"] = reply.strip()
        print("Team:", self.state["team"], "| Priority:", self.state["priority"])
        print("Reply:", self.state["resolution"])
        return self.state["resolution"]

In [ ]:
tickets = [
    "I was charged twice for my October subscription",
    "How do I download last month's invoice?",
    "My account was hacked and I can't log in",
    "Can you please add a dark mode to the app?"
    "Can you please update my refund status?",
]

for t in tickets:
    flow = SupportFlow()
    await flow.kickoff_async(inputs={"ticket": t})

╭───────────────────────────── 🌊 Flow Execution ──────────────────────────────╮
│                                                                              │
│  Starting Flow Execution                                                     │
│  Name: SupportFlow                                                           │
│  ID: 7d9650ea-3eda-4f32-96aa-43a08f8eac7d                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 🌊 Flow Started ───────────────────────────────╮
│                                                                              │
│  Flow Started                                                                │
│  Name: SupportFlow                                                           │
│  ID: 7d9650ea-3eda-4f32-96a